# 08 promotion vs non-promotion EDA 260513

Step 08 descriptive EDA only. This notebook compares promotion rows and non-promotion rows inside the step 06 primary main cohort using conservative features only. No modeling, prediction, SHAP, Optuna, p-values, statistical significance testing, feature engineering, additional row exclusion, or causal inference is performed.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import zipfile

import numpy as np
import pandas as pd

STEP_NAME = '08_promotion_vs_nonpromotion_eda_260513'
EXPECTED_REPO_ROOTS = ['C:/Code/ott-churn-prediction', 'C:\\Code\\ott-churn-prediction']
EXPECTED_RAW_ROWS = 23343
EXPECTED_RAW_COLS = 91
EXPECTED_MAIN_ROWS = 23079
EXPECTED_CONSERVATIVE_FEATURES = 22

def is_inside(child, parent):
    try:
        Path(child).resolve().relative_to(Path(parent).resolve())
        return True
    except ValueError:
        return False

def write_csv(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding='utf-8-sig')

def yn(value):
    return 'yes' if bool(value) else 'no'

def safe_join(values):
    vals = [str(v) for v in values if pd.notna(v) and str(v) != '']
    return ';'.join(vals)

def q(series, p):
    s = pd.to_numeric(series, errors='coerce').dropna()
    return np.nan if len(s) == 0 else float(s.quantile(p))

def rate(mask):
    return np.nan if len(mask) == 0 else float(np.mean(mask))

actual_repo_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()
repo_root_match = actual_repo_root in EXPECTED_REPO_ROOTS
print('actual_repo_root:', actual_repo_root)
print('repo_root_match:', repo_root_match)
if not repo_root_match:
    raise SystemExit('STOP: repo root mismatch. No files were written.')

ROOT = Path(actual_repo_root).resolve()
PARK = ROOT / 'park.ingyeom'
SOURCE = PARK / 'data' / '(광일)Membership_v2_with_derived_features.csv'
NOTE = PARK / 'note.md'
NOTEBOOK_PATH = PARK / 'notebook' / STEP_NAME / f'{STEP_NAME}.ipynb'
OUTPUT_BASE = PARK / 'reports' / 'eda' / STEP_NAME
ZIP_DIR = PARK / 'zip'
ZIP_PATH = ZIP_DIR / f'{STEP_NAME}_review_package.zip'

previous_final_checks = {
    '01': PARK / 'reports' / 'audits' / '01_data_contract_260513' / '01_final_checks.csv',
    '02': PARK / 'reports' / 'audits' / '02_target_score_orientation_260513' / '02_final_checks.csv',
    '03': PARK / 'reports' / 'audits' / '03_observation_window_policy_260513' / '03_final_checks.csv',
    '04': PARK / 'reports' / 'audits' / '04_promotion_split_260513' / '04_final_checks.csv',
    '05b': PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_final_checks.csv',
    '06': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_final_checks.csv',
    '07': PARK / 'reports' / 'audits' / '07_AARRR_feature_mapping_260513' / '07_final_checks.csv',
}
files_06 = {
    'index': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_primary_main_cohort_index.csv',
    'conservative': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_primary_main_cohort_conservative_features.csv',
    'cohort_summary': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_cohort_summary.csv',
    'before_after': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_before_after_distribution_impact.csv',
    'feature_policy': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_feature_policy_from_05b.csv',
    'open_risks': PARK / 'reports' / 'audits' / '06_common_preprocessing_and_final_cohort_260513' / '06_open_risks_for_next_steps.csv',
}
files_05b = {
    'dictionary': PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_canonical_column_role_dictionary.csv',
    'safe': PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_conservative_safe_candidate_columns.csv',
    'review': PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_review_required_columns.csv',
    'forbidden': PARK / 'reports' / 'audits' / '05b_column_role_dictionary_patch_260513' / '05b_forbidden_drop_columns.csv',
}
files_07 = {
    'all_mapping': PARK / 'reports' / 'audits' / '07_AARRR_feature_mapping_260513' / '07_AARRR_feature_mapping_all_columns.csv',
    'cons_mapping': PARK / 'reports' / 'audits' / '07_AARRR_feature_mapping_260513' / '07_AARRR_mapping_conservative_features.csv',
    'limitations': PARK / 'reports' / 'audits' / '07_AARRR_feature_mapping_260513' / '07_AARRR_measurement_limitations.csv',
    'eda_plan': PARK / 'reports' / 'audits' / '07_AARRR_feature_mapping_260513' / '07_AARRR_to_EDA_plan.csv',
    'ladder_handoff': PARK / 'reports' / 'audits' / '07_AARRR_feature_mapping_260513' / '07_AARRR_to_baseline_ladder_handoff.csv',
    'open_risks': PARK / 'reports' / 'audits' / '07_AARRR_feature_mapping_260513' / '07_open_risks_for_next_steps.csv',
}

if OUTPUT_BASE.exists() and any(OUTPUT_BASE.iterdir()):
    OUTPUT_DIR = OUTPUT_BASE / ('run_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
else:
    OUTPUT_DIR = OUTPUT_BASE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

preflight = {
    'expected_repo_root': 'C:/Code/ott-churn-prediction or C:\\Code\\ott-churn-prediction',
    'actual_repo_root': actual_repo_root,
    'repo_root_match': repo_root_match,
    'source_file_exists': SOURCE.exists(),
    'previous_final_checks_exist_01': previous_final_checks['01'].exists(),
    'previous_final_checks_exist_02': previous_final_checks['02'].exists(),
    'previous_final_checks_exist_03': previous_final_checks['03'].exists(),
    'previous_final_checks_exist_04': previous_final_checks['04'].exists(),
    'previous_final_checks_exist_05b': previous_final_checks['05b'].exists(),
    'previous_final_checks_exist_06': previous_final_checks['06'].exists(),
    'previous_final_checks_exist_07': previous_final_checks['07'].exists(),
    'required_06_files_exist': all(p.exists() for p in files_06.values()),
    'required_05b_files_exist': all(p.exists() for p in files_05b.values()),
    'required_07_files_exist': all(p.exists() for p in files_07.values()),
    'note_md_exists': NOTE.exists(),
    'source_file_inside_park_ingyeom': is_inside(SOURCE, PARK),
    'output_folder_inside_park_ingyeom': is_inside(OUTPUT_DIR, PARK),
    'notebook_inside_park_ingyeom': is_inside(NOTEBOOK_PATH, PARK),
    'zip_folder_inside_park_ingyeom': is_inside(ZIP_DIR, PARK),
}
preflight['can_proceed'] = all(bool(v) for k, v in preflight.items() if k not in ['expected_repo_root', 'actual_repo_root'])
write_csv(pd.DataFrame([{'check_name': k, 'value': v} for k, v in preflight.items()]), OUTPUT_DIR / '08_preflight_input_validation.csv')
if not preflight['can_proceed']:
    missing = [str(p.relative_to(PARK)) for p in [SOURCE, NOTE] + list(previous_final_checks.values()) + list(files_06.values()) + list(files_05b.values()) + list(files_07.values()) if not p.exists()]
    readme = ['# 08 promotion vs non-promotion EDA', '', 'This run stopped at preflight validation.', '', 'No downstream EDA outputs were created.', '', 'Missing or failed inputs:']
    readme.extend([f'- {m}' for m in missing] or ['- See 08_preflight_input_validation.csv'])
    (OUTPUT_DIR / 'README.md').write_text('\n'.join(readme) + '\n', encoding='utf-8')
    raise SystemExit('STOP: preflight failed. Only validation and README were written.')

source_stat_before = SOURCE.stat()
raw = pd.read_csv(SOURCE, encoding='utf-8-sig')
cohort_index = pd.read_csv(files_06['index'], encoding='utf-8-sig')
cons = pd.read_csv(files_06['conservative'], encoding='utf-8-sig')
dict05 = pd.read_csv(files_05b['dictionary'], encoding='utf-8-sig')
safe05 = pd.read_csv(files_05b['safe'], encoding='utf-8-sig')
review05 = pd.read_csv(files_05b['review'], encoding='utf-8-sig')
forbidden05 = pd.read_csv(files_05b['forbidden'], encoding='utf-8-sig')
map_all07 = pd.read_csv(files_07['all_mapping'], encoding='utf-8-sig')
map_cons07 = pd.read_csv(files_07['cons_mapping'], encoding='utf-8-sig')
limitations07 = pd.read_csv(files_07['limitations'], encoding='utf-8-sig')

review_set = set(review05['column_name'].astype(str))
forbidden_set = set(forbidden05['column_name'].astype(str))
flag_cols = [c for c in cons.columns if c.startswith('flag_')]
metadata_cols = {'source_row_number', 'USER_KEY', 'is_promotion', 'is_repurchase', 'duration_days'} | set(flag_cols)
feature_cols = [c for c in cons.columns if c not in metadata_cols]
actual_map_cons = map_cons07[~map_cons07['column_name'].astype(str).str.startswith('__summary')].copy()
map_info = actual_map_cons.set_index('column_name').to_dict('index')
prediction_like_cols = [c for c in cons.columns if 'predict' in c.lower() or c.lower().endswith('_pred') or c.lower().startswith('pred_') or 'prediction' in c.lower()]

def check_status(condition):
    return 'PASS' if bool(condition) else 'MISMATCH'

cohort_checks = pd.DataFrame([
    {'check_name': 'raw source rows', 'actual_value': len(raw), 'expected_value': EXPECTED_RAW_ROWS, 'status': check_status(len(raw) == EXPECTED_RAW_ROWS), 'detail': ''},
    {'check_name': 'raw source columns', 'actual_value': len(raw.columns), 'expected_value': EXPECTED_RAW_COLS, 'status': check_status(len(raw.columns) == EXPECTED_RAW_COLS), 'detail': ''},
    {'check_name': 'primary main cohort index rows', 'actual_value': len(cohort_index), 'expected_value': EXPECTED_MAIN_ROWS, 'status': check_status(len(cohort_index) == EXPECTED_MAIN_ROWS), 'detail': ''},
    {'check_name': 'conservative feature table rows', 'actual_value': len(cons), 'expected_value': EXPECTED_MAIN_ROWS, 'status': check_status(len(cons) == EXPECTED_MAIN_ROWS), 'detail': ''},
    {'check_name': 'conservative actual feature count', 'actual_value': len(feature_cols), 'expected_value': EXPECTED_CONSERVATIVE_FEATURES, 'status': check_status(len(feature_cols) == EXPECTED_CONSERVATIVE_FEATURES), 'detail': safe_join(feature_cols)},
    {'check_name': 'no review columns appear as conservative feature columns', 'actual_value': len(set(feature_cols) & review_set), 'expected_value': 0, 'status': check_status(len(set(feature_cols) & review_set) == 0), 'detail': safe_join(sorted(set(feature_cols) & review_set))},
    {'check_name': 'no forbidden columns appear as conservative feature columns', 'actual_value': len(set(feature_cols) & forbidden_set), 'expected_value': 0, 'status': check_status(len(set(feature_cols) & forbidden_set) == 0), 'detail': safe_join(sorted(set(feature_cols) & forbidden_set))},
    {'check_name': 'is_promotion exists in conservative table as split metadata', 'actual_value': yn('is_promotion' in cons.columns), 'expected_value': 'yes', 'status': check_status('is_promotion' in cons.columns), 'detail': ''},
    {'check_name': 'is_repurchase exists in conservative table as target', 'actual_value': yn('is_repurchase' in cons.columns), 'expected_value': 'yes', 'status': check_status('is_repurchase' in cons.columns), 'detail': ''},
    {'check_name': 'USER_KEY exists as metadata/group key', 'actual_value': yn('USER_KEY' in cons.columns), 'expected_value': 'yes', 'status': check_status('USER_KEY' in cons.columns), 'detail': ''},
    {'check_name': 'no repurchase_score column exists', 'actual_value': yn('repurchase_score' not in cons.columns), 'expected_value': 'yes', 'status': check_status('repurchase_score' not in cons.columns), 'detail': ''},
    {'check_name': 'no churn_risk column exists', 'actual_value': yn('churn_risk' not in cons.columns), 'expected_value': 'yes', 'status': check_status('churn_risk' not in cons.columns), 'detail': ''},
    {'check_name': 'no model prediction columns exist', 'actual_value': len(prediction_like_cols), 'expected_value': 0, 'status': check_status(len(prediction_like_cols) == 0), 'detail': safe_join(prediction_like_cols)},
    {'check_name': '07 conservative mapping summary rows ignored as features', 'actual_value': len(actual_map_cons), 'expected_value': EXPECTED_CONSERVATIVE_FEATURES, 'status': check_status(len(actual_map_cons) == EXPECTED_CONSERVATIVE_FEATURES), 'detail': 'Rows with column_name starting __summary were excluded.'},
])
write_csv(cohort_checks, OUTPUT_DIR / '08_cohort_consistency_check.csv')

def group_overview(df, label, mask=None):
    sub = df if mask is None else df.loc[mask].copy()
    dur = pd.to_numeric(sub['duration_days'], errors='coerce')
    return {
        'is_promotion': label,
        'row_count': len(sub),
        'row_rate': len(sub) / len(df) if len(df) else np.nan,
        'unique_USER_KEY_count': sub['USER_KEY'].nunique(dropna=False),
        'duplicated_USER_KEY_extra_rows': len(sub) - sub['USER_KEY'].nunique(dropna=False),
        'cross_promotion_USER_KEY_overlap_rows': int(sub['flag_cross_promotion_USER_KEY_overlap'].sum()) if 'flag_cross_promotion_USER_KEY_overlap' in sub else 0,
        'repurchase_count': int((sub['is_repurchase'] == 1).sum()),
        'nonrepurchase_count': int((sub['is_repurchase'] == 0).sum()),
        'repurchase_rate': float((sub['is_repurchase'] == 1).mean()) if len(sub) else np.nan,
        'nonrepurchase_rate': float((sub['is_repurchase'] == 0).mean()) if len(sub) else np.nan,
        'duration_min': dur.min(),
        'duration_max': dur.max(),
        'duration_mean': dur.mean(),
        'duration_median': dur.median(),
        'duration_q25': dur.quantile(0.25),
        'duration_q75': dur.quantile(0.75),
    }

overview_rows = [group_overview(cons, 'overall')]
for val in sorted(cons['is_promotion'].dropna().unique()):
    overview_rows.append(group_overview(cons, int(val), cons['is_promotion'] == val))
overview = pd.DataFrame(overview_rows)
write_csv(overview, OUTPUT_DIR / '08_promotion_group_overview.csv')

ct = cons.groupby(['is_promotion', 'is_repurchase']).size().reset_index(name='count')
ct['percent_of_total'] = ct['count'] / len(cons)
ct['percent_within_promotion_group'] = ct['count'] / ct.groupby('is_promotion')['count'].transform('sum')
ct['percent_within_target_group'] = ct['count'] / ct.groupby('is_repurchase')['count'].transform('sum')
write_csv(ct, OUTPUT_DIR / '08_promotion_target_2x2_main_cohort.csv')

nonpromo_rate = float(cons.loc[cons['is_promotion'] == 0, 'is_repurchase'].mean())
promo_rate = float(cons.loc[cons['is_promotion'] == 1, 'is_repurchase'].mean())
rate_diff = pd.DataFrame([{
    'nonpromotion_repurchase_rate': nonpromo_rate,
    'promotion_repurchase_rate': promo_rate,
    'absolute_difference_promotion_minus_nonpromotion': promo_rate - nonpromo_rate,
    'percentage_point_difference': promo_rate - nonpromo_rate,
    'relative_difference_descriptive_only': (promo_rate / nonpromo_rate - 1) if nonpromo_rate else np.nan,
    'safe_interpretation': 'This is descriptive only. Primary main cohort shows a repurchase-rate difference between promotion and non-promotion rows.',
    'unsafe_interpretation': 'Do not infer causal effect or claim promotion caused the difference.',
}])
write_csv(rate_diff, OUTPUT_DIR / '08_promotion_repurchase_rate_difference.csv')

def dataset_summary(name, frame):
    promo = frame['is_promotion']
    target = frame['is_repurchase']
    rows = [
        {'dataset': name, 'metric': 'total rows', 'value': len(frame), 'difference_caused_by_06_row_policy': '', 'interpretation_note': ''},
        {'dataset': name, 'metric': 'promotion count', 'value': int((promo == 1).sum()), 'difference_caused_by_06_row_policy': '', 'interpretation_note': ''},
        {'dataset': name, 'metric': 'promotion rate', 'value': float((promo == 1).mean()), 'difference_caused_by_06_row_policy': '', 'interpretation_note': 'descriptive split only'},
        {'dataset': name, 'metric': 'nonpromotion count', 'value': int((promo == 0).sum()), 'difference_caused_by_06_row_policy': '', 'interpretation_note': ''},
        {'dataset': name, 'metric': 'nonpromotion rate', 'value': float((promo == 0).mean()), 'difference_caused_by_06_row_policy': '', 'interpretation_note': ''},
        {'dataset': name, 'metric': 'repurchase count', 'value': int((target == 1).sum()), 'difference_caused_by_06_row_policy': '', 'interpretation_note': 'target proxy, not actual revenue'},
        {'dataset': name, 'metric': 'repurchase rate', 'value': float((target == 1).mean()), 'difference_caused_by_06_row_policy': '', 'interpretation_note': ''},
        {'dataset': name, 'metric': 'nonrepurchase count', 'value': int((target == 0).sum()), 'difference_caused_by_06_row_policy': '', 'interpretation_note': ''},
        {'dataset': name, 'metric': 'nonrepurchase rate', 'value': float((target == 0).mean()), 'difference_caused_by_06_row_policy': '', 'interpretation_note': ''},
    ]
    for _, r in pd.crosstab(frame['is_promotion'], frame['is_repurchase']).stack().reset_index(name='count').iterrows():
        rows.append({'dataset': name, 'metric': f'promotion_{r.is_promotion}_target_{r.is_repurchase}_2x2_count', 'value': int(r['count']), 'difference_caused_by_06_row_policy': '', 'interpretation_note': 'promotion by target 2x2 count'})
    return rows
raw_main_rows = dataset_summary('raw_source', raw) + dataset_summary('primary_main_cohort', cons)
raw_main = pd.DataFrame(raw_main_rows)
raw_metrics = raw_main[raw_main['dataset'] == 'raw_source'].set_index('metric')['value']
main_metrics = raw_main[raw_main['dataset'] == 'primary_main_cohort'].set_index('metric')['value']
raw_main['difference_caused_by_06_row_policy'] = raw_main.apply(lambda r: '' if r['dataset'] == 'raw_source' or r['metric'] not in raw_metrics.index else r['value'] - raw_metrics.get(r['metric'], np.nan), axis=1)
write_csv(raw_main, OUTPUT_DIR / '08_raw_vs_main_promotion_target_comparison.csv')

def feature_stats(frame, feature, group_cols):
    out = []
    for keys, sub in frame.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        s = pd.to_numeric(sub[feature], errors='coerce')
        rec = {col: keys[i] for i, col in enumerate(group_cols)}
        rec.update({
            'feature_name': feature,
            'AARRR_stage_primary': map_info.get(feature, {}).get('AARRR_stage_primary', ''),
            'feature_family': map_info.get(feature, {}).get('feature_family', ''),
            'n': len(sub),
            'missing_count': int(s.isna().sum()),
            'mean': s.mean(),
            'std': s.std(ddof=1),
            'min': s.min(),
            'q10': s.quantile(0.10),
            'q25': s.quantile(0.25),
            'median': s.median(),
            'q75': s.quantile(0.75),
            'q90': s.quantile(0.90),
            'max': s.max(),
            'zero_count': int((s == 0).sum()),
            'zero_rate': float((s == 0).mean()) if len(s) else np.nan,
            'positive_count': int((s > 0).sum()),
            'positive_rate': float((s > 0).mean()) if len(s) else np.nan,
        })
        out.append(rec)
    return out

dist_rows = []
for feature in feature_cols:
    dist_rows.extend(feature_stats(cons, feature, ['is_promotion']))
dist_promo = pd.DataFrame(dist_rows)
write_csv(dist_promo, OUTPUT_DIR / '08_conservative_feature_distribution_by_promotion.csv')

def smd_for_feature(feature, mask_a, mask_b):
    a = pd.to_numeric(cons.loc[mask_a, feature], errors='coerce').dropna()
    b = pd.to_numeric(cons.loc[mask_b, feature], errors='coerce').dropna()
    if len(a) < 2 or len(b) < 2:
        return np.nan
    pooled = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return np.nan if pooled == 0 or pd.isna(pooled) else float((b.mean() - a.mean()) / pooled)

def bucket_smd(v):
    if pd.isna(v):
        return 'not_computable'
    av = abs(v)
    if av < 0.10:
        return 'negligible'
    if av < 0.30:
        return 'small'
    if av < 0.50:
        return 'medium'
    return 'large'

diff_rows = []
for feature in feature_cols:
    non = pd.to_numeric(cons.loc[cons['is_promotion'] == 0, feature], errors='coerce')
    pro = pd.to_numeric(cons.loc[cons['is_promotion'] == 1, feature], errors='coerce')
    smd = smd_for_feature(feature, cons['is_promotion'] == 0, cons['is_promotion'] == 1)
    diff_rows.append({
        'feature_name': feature,
        'AARRR_stage_primary': map_info.get(feature, {}).get('AARRR_stage_primary', ''),
        'feature_family': map_info.get(feature, {}).get('feature_family', ''),
        'nonpromotion_mean': non.mean(),
        'promotion_mean': pro.mean(),
        'mean_difference_promotion_minus_nonpromotion': pro.mean() - non.mean(),
        'nonpromotion_median': non.median(),
        'promotion_median': pro.median(),
        'median_difference': pro.median() - non.median(),
        'nonpromotion_zero_rate': float((non == 0).mean()),
        'promotion_zero_rate': float((pro == 0).mean()),
        'zero_rate_difference': float((pro == 0).mean()) - float((non == 0).mean()),
        'simple_standardized_mean_difference': smd,
        'descriptive_effect_size_bucket': bucket_smd(smd),
        'safe_interpretation': 'Descriptive effect size only, not statistical significance and not causal.',
        'caution': 'No p-values were created. Review columns remain excluded.'
    })
diff_summary = pd.DataFrame(diff_rows)
write_csv(diff_summary, OUTPUT_DIR / '08_conservative_feature_promotion_difference_summary.csv')

dist_pt_rows = []
for feature in feature_cols:
    for keys, sub in cons.groupby(['is_promotion', 'is_repurchase'], dropna=False):
        s = pd.to_numeric(sub[feature], errors='coerce')
        dist_pt_rows.append({
            'feature_name': feature,
            'AARRR_stage_primary': map_info.get(feature, {}).get('AARRR_stage_primary', ''),
            'feature_family': map_info.get(feature, {}).get('feature_family', ''),
            'is_promotion': keys[0],
            'is_repurchase': keys[1],
            'n': len(sub),
            'mean': s.mean(),
            'std': s.std(ddof=1),
            'median': s.median(),
            'q25': s.quantile(0.25),
            'q75': s.quantile(0.75),
            'zero_rate': float((s == 0).mean()) if len(s) else np.nan,
            'positive_rate': float((s > 0).mean()) if len(s) else np.nan,
        })
dist_pt = pd.DataFrame(dist_pt_rows)
write_csv(dist_pt, OUTPUT_DIR / '08_conservative_feature_distribution_by_promotion_target.csv')

top_rows = []
for stage in sorted(diff_summary['AARRR_stage_primary'].dropna().unique()):
    stage_features = diff_summary[diff_summary['AARRR_stage_primary'] == stage].copy()
    top_promo = stage_features.reindex(stage_features['simple_standardized_mean_difference'].abs().sort_values(ascending=False).index).head(5)
    top_rows.append({'AARRR_stage_primary': stage, 'comparison_type': 'promotion_vs_nonpromotion_abs_smd', 'top_features': safe_join(top_promo['feature_name']), 'top_metric_values': safe_join(top_promo['simple_standardized_mean_difference'].round(6)), 'descriptive_only_note': 'Effect-size ranking only; no p-values and no causal claim.'})
    for promo_val, label in [(1, 'repurchase_vs_nonrepurchase_within_promotion'), (0, 'repurchase_vs_nonrepurchase_within_nonpromotion')]:
        rows = []
        for feature in stage_features['feature_name']:
            smd_t = smd_for_feature(feature, (cons['is_promotion'] == promo_val) & (cons['is_repurchase'] == 0), (cons['is_promotion'] == promo_val) & (cons['is_repurchase'] == 1))
            rows.append({'feature': feature, 'smd': smd_t})
        tmp = pd.DataFrame(rows).sort_values('smd', key=lambda s: s.abs(), ascending=False).head(5)
        top_rows.append({'AARRR_stage_primary': stage, 'comparison_type': label, 'top_features': safe_join(tmp['feature']), 'top_metric_values': safe_join(tmp['smd'].round(6)), 'descriptive_only_note': 'Within-group target difference ranking only; target is repurchase proxy.'})
write_csv(pd.DataFrame(top_rows), OUTPUT_DIR / '08_top_descriptive_differences_by_AARRR_stage.csv')

aarr_rows = []
for stage in ['acquisition', 'activation', 'retention', 'revenue_proxy', 'referral']:
    feats = [f for f in feature_cols if map_info.get(f, {}).get('AARRR_stage_primary') == stage]
    if feats:
        avg_abs_smd = diff_summary.loc[diff_summary['feature_name'].isin(feats), 'simple_standardized_mean_difference'].abs().mean()
        non_avg = np.nanmean([pd.to_numeric(cons.loc[cons['is_promotion'] == 0, f], errors='coerce').mean() for f in feats])
        pro_avg = np.nanmean([pd.to_numeric(cons.loc[cons['is_promotion'] == 1, f], errors='coerce').mean() for f in feats])
        pattern = f'{len(feats)} conservative feature(s) available; descriptive mean comparison only.'
        caution = 'No p-values, no causal inference.'
    elif stage == 'acquisition':
        avg_abs_smd = non_avg = pro_avg = np.nan
        pattern = 'Acquisition is represented by is_promotion split metadata, not ordinary conservative features.'
        caution = 'Descriptive split only, not causal effect.'
    elif stage == 'revenue_proxy':
        avg_abs_smd = non_avg = pro_avg = np.nan
        pattern = 'Revenue proxy is is_repurchase target, not ordinary feature.'
        caution = 'Not actual revenue amount.'
    elif stage == 'referral':
        avg_abs_smd = non_avg = pro_avg = np.nan
        pattern = 'Referral is not observed in current data.'
        caution = 'Future experiment proposal only.'
    else:
        avg_abs_smd = non_avg = pro_avg = np.nan
        pattern = 'No conservative feature available for this stage.'
        caution = 'Do not fill with review columns.'
    aarr_rows.append({'AARRR_stage': stage, 'number_of_features': len(feats), 'feature_names': safe_join(feats), 'nonpromotion_feature_mean_average_if_meaningful': non_avg, 'promotion_feature_mean_average_if_meaningful': pro_avg, 'average_absolute_standardized_mean_difference_if_computable': avg_abs_smd, 'key_descriptive_pattern': pattern, 'caution': caution})
write_csv(pd.DataFrame(aarr_rows), OUTPUT_DIR / '08_AARRR_summary_by_promotion.csv')

flag_rows = []
flag_candidate_cols = [c for c in cohort_index.columns if c.startswith('flag_')]
for promo_val, sub in cohort_index.groupby('is_promotion'):
    dur = pd.to_numeric(sub['duration_days'], errors='coerce')
    rec = {'is_promotion': promo_val, 'row_count': len(sub), 'duration_min': dur.min(), 'duration_max': dur.max(), 'duration_mean': dur.mean(), 'duration_median': dur.median(), 'duration_q25': dur.quantile(0.25), 'duration_q75': dur.quantile(0.75)}
    for c in flag_candidate_cols:
        rec[f'{c}_count'] = int(pd.to_numeric(sub[c], errors='coerce').fillna(0).sum())
        rec[f'{c}_rate'] = float(pd.to_numeric(sub[c], errors='coerce').fillna(0).mean())
    flag_rows.append(rec)
write_csv(pd.DataFrame(flag_rows), OUTPUT_DIR / '08_flag_distribution_by_promotion.csv')

review_rows = []
for _, row in review05.iterrows():
    col = row['column_name']
    review_rows.append({
        'column_name': col,
        'patched_primary_role': row.get('patched_primary_role', ''),
        'patched_feature_family': row.get('patched_feature_family', ''),
        'why_excluded_from_standard_eda': '05b marks this column as review-required; step 08 conservative EDA uses only 06 conservative safe features.',
        'possible_future_resolution': row.get('patched_future_step_to_resolve', row.get('future_step_to_resolve', 'explicit review required')),
        'possible_sensitivity_use': 'May be used only in a later review-resolved or sensitivity analysis, not standard step 08 EDA.',
    })
write_csv(pd.DataFrame(review_rows), OUTPUT_DIR / '08_review_columns_excluded_from_standard_eda.csv')

top3 = diff_summary.reindex(diff_summary['simple_standardized_mean_difference'].abs().sort_values(ascending=False).index).head(3)
findings = pd.DataFrame([
    {'finding_area': 'group_size', 'descriptive_finding': f'Promotion rows={int((cons.is_promotion==1).sum())}; non-promotion rows={int((cons.is_promotion==0).sum())}.', 'strength_of_evidence': 'strong', 'safe_claim': 'Group sizes differ descriptively in the primary main cohort.', 'forbidden_claim': 'Group size difference is caused by promotion.', 'next_check_needed': 'Use row-level language.'},
    {'finding_area': 'repurchase_rate', 'descriptive_finding': f'Promotion repurchase rate={promo_rate:.6f}; non-promotion repurchase rate={nonpromo_rate:.6f}.', 'strength_of_evidence': 'strong', 'safe_claim': 'A descriptive repurchase-rate difference is observed.', 'forbidden_claim': 'Promotion caused the repurchase-rate difference.', 'next_check_needed': 'Deeper 2x2 EDA in step 09.'},
    {'finding_area': 'duration_distribution', 'descriptive_finding': 'Duration summaries by promotion are provided in group overview and flag tables.', 'strength_of_evidence': 'moderate', 'safe_claim': 'Duration can be compared descriptively inside the main cohort.', 'forbidden_claim': 'Duration proves behavioral causality.', 'next_check_needed': 'Check anomaly context separately.'},
    {'finding_area': 'activation_features', 'descriptive_finding': f'{sum(diff_summary.AARRR_stage_primary == "activation")} conservative activation features summarized.', 'strength_of_evidence': 'moderate', 'safe_claim': 'Early viewing proxies are available for descriptive comparison.', 'forbidden_claim': 'Activation quality or satisfaction is fully measured.', 'next_check_needed': 'Step 10 feature distribution deep dive.'},
    {'finding_area': 'retention_features', 'descriptive_finding': f'{sum(diff_summary.AARRR_stage_primary == "retention")} conservative retention features summarized.', 'strength_of_evidence': 'moderate', 'safe_claim': 'Week1-3 retention proxies are available descriptively.', 'forbidden_claim': 'Long-term retention is fully validated.', 'next_check_needed': 'Step 10 feature distribution deep dive.'},
    {'finding_area': 'duplicated_USER_KEY/cross-promotion overlap', 'descriptive_finding': 'Flag distribution table preserves duplicate and overlap cautions.', 'strength_of_evidence': 'structural', 'safe_claim': 'Rows remain subscription-event-level.', 'forbidden_claim': 'Rows are unique users.', 'next_check_needed': 'Maintain group-aware CV later.'},
    {'finding_area': 'review_columns_excluded', 'descriptive_finding': f'{len(review05)} review columns excluded from standard EDA.', 'strength_of_evidence': 'structural', 'safe_claim': 'Review columns are separated for later resolution.', 'forbidden_claim': 'Excluding review columns means no information loss.', 'next_check_needed': 'Sensitivity design if needed.'},
    {'finding_area': 'AARRR_coverage', 'descriptive_finding': 'Conservative features cover activation and retention; acquisition is split metadata; revenue is target proxy; referral is not observed.', 'strength_of_evidence': 'structural', 'safe_claim': 'AARRR coverage is partial and conservative.', 'forbidden_claim': 'AARRR is fully validated from current data.', 'next_check_needed': 'Use 07/08 handoff.'},
    {'finding_area': 'causal_limitations', 'descriptive_finding': 'No causal inference, A/B testing, p-values, or modeling performed.', 'strength_of_evidence': 'structural', 'safe_claim': 'Findings are descriptive only.', 'forbidden_claim': 'Promotion caused any observed difference.', 'next_check_needed': 'Keep safe wording.'},
    {'finding_area': 'next_step', 'descriptive_finding': 'Proceed to 09_promotion_repurchase_2x2_eda_260513.', 'strength_of_evidence': 'structural', 'safe_claim': 'Next step deepens promotion x repurchase structure.', 'forbidden_claim': 'Skip directly to modeling without checking 2x2 and feature distributions.', 'next_check_needed': 'Step 09.'},
])
write_csv(findings, OUTPUT_DIR / '08_descriptive_findings_summary.csv')

safe_unsafe = pd.DataFrame([
    {'unsafe_wording': '100원딜 때문에 재구매율이 낮아졌다.', 'safer_wording': 'primary main cohort에서 프로모션 행과 비프로모션 행 사이에 재구매율 차이가 관찰되었다.'},
    {'unsafe_wording': '프로모션 고객은 이탈 성향이 높다.', 'safer_wording': '프로모션 행은 비프로모션 행보다 낮은 재구매율이 관찰되었지만, 이는 인과효과나 고객 본질 성향을 의미하지 않는다.'},
    {'unsafe_wording': 'feature 차이가 유의하다.', 'safer_wording': '이 단계는 p-value 검정을 하지 않은 descriptive EDA이며, 차이는 분포와 효과크기 관점의 탐색 신호다.'},
    {'unsafe_wording': 'Referral 분석 결과가 나왔다.', 'safer_wording': 'Referral은 현재 데이터에서 관측되지 않아 후속 실험 제안으로만 둔다.'},
    {'unsafe_wording': 'review 컬럼을 제외했으니 정보 손실이 없다.', 'safer_wording': '보수적 표준 분석을 위해 review 컬럼을 제외했으며, 정보 손실 가능성은 별도 sensitivity 실험에서 다룰 수 있다.'},
])
write_csv(safe_unsafe, OUTPUT_DIR / '08_safe_unsafe_wording.csv')

open_risks = pd.DataFrame({'risk_to_carry_forward': [
    'promotion/non-promotion differences remain descriptive, not causal',
    'review columns remain excluded from standard EDA and modeling',
    'membership/context L0 baseline may require separate review-resolution or sensitivity design',
    'content/genre ratio insights are limited until observation window is confirmed',
    'total/all-period and recency columns remain unresolved',
    'duplicated USER_KEY and cross-promotion overlap require row-level language',
    'Referral is not observed',
    '09 should examine promotion x repurchase 2x2 structure more deeply',
    '10 should examine feature distributions more deeply before modeling',
    '11 baseline ladder should not use review columns without explicit decision',
]})
write_csv(open_risks, OUTPUT_DIR / '08_open_risks_for_next_steps.csv')

created_csv_names = [
    '08_preflight_input_validation.csv', '08_cohort_consistency_check.csv', '08_promotion_group_overview.csv',
    '08_promotion_target_2x2_main_cohort.csv', '08_promotion_repurchase_rate_difference.csv',
    '08_raw_vs_main_promotion_target_comparison.csv', '08_conservative_feature_distribution_by_promotion.csv',
    '08_conservative_feature_promotion_difference_summary.csv', '08_conservative_feature_distribution_by_promotion_target.csv',
    '08_top_descriptive_differences_by_AARRR_stage.csv', '08_AARRR_summary_by_promotion.csv',
    '08_flag_distribution_by_promotion.csv', '08_review_columns_excluded_from_standard_eda.csv',
    '08_descriptive_findings_summary.csv', '08_safe_unsafe_wording.csv', '08_open_risks_for_next_steps.csv',
    '08_final_checks.csv'
]

readme = f'''# {STEP_NAME}

This is step 08 only.

- This is descriptive EDA only.
- No modeling was performed.
- No predictions were created.
- No repurchase_score or churn_risk was created.
- No SHAP was performed.
- No Optuna was performed.
- No statistical significance testing was performed.
- No p-values were created.
- No feature engineering was performed.
- No additional row exclusion was performed.
- Review columns were not used in standard conservative feature EDA.
- Differences are descriptive only and not causal.
- Referral is not observed.

Next recommended step is 09_promotion_repurchase_2x2_eda_260513.
'''
(OUTPUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

now_text = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
row_counts = {str(k): int(v) for k, v in overview.set_index('is_promotion')['row_count'].to_dict().items()}
rate_by_promo = {str(k): float(v) for k, v in overview.set_index('is_promotion')['repurchase_rate'].to_dict().items()}
note_section = f'''

## {now_text} | {STEP_NAME}

- Purpose: primary main cohort 안에서 promotion/non-promotion 행을 보수 feature 기준으로 descriptive EDA 비교했다.
- Files created: {len(created_csv_names)} CSV files, README.md, notebook, review package zip.
- Key descriptive findings: promotion/non-promotion row count와 repurchase rate 차이를 관찰했고, conservative feature 22개에 대해 분포와 단순 표준화 평균 차이를 계산했다.
- Promotion/non-promotion row counts: {json.dumps(row_counts, ensure_ascii=False, sort_keys=True)}
- Repurchase rates by promotion: {json.dumps(rate_by_promo, ensure_ascii=False, sort_keys=True)}
- Conservative feature count: {len(feature_cols)}
- Review columns policy: 05b review columns은 standard conservative EDA feature table에서 제외했다.
- Checks passed or failed: final checks table 참조.
- Interpretation limits: descriptive only, no p-values, no statistical testing, no modeling, no causal claim, no Referral measurement.
- Risks to carry forward: promotion 차이는 인과가 아니며, duplicated USER_KEY/cross-promotion overlap은 row-level 언어로 유지해야 한다. review columns은 후속 resolution 또는 sensitivity design 필요.
- Next step recommendation: 09_promotion_repurchase_2x2_eda_260513.
'''
with NOTE.open('a', encoding='utf-8') as f:
    f.write(note_section)

source_stat_after = SOURCE.stat()
generated_paths = [OUTPUT_DIR / n for n in created_csv_names if n != '08_final_checks.csv'] + [OUTPUT_DIR / 'README.md', NOTE, ZIP_PATH, NOTEBOOK_PATH]
no_files_outside_park = all(is_inside(p, PARK) for p in generated_paths)

def check_row(name, condition, detail=''):
    return {'check_name': name, 'status': 'PASS' if bool(condition) else 'FAIL', 'detail': detail}

checks = [
    check_row('repo_root_checked', True, actual_repo_root),
    check_row('repo_root_matches_expected', repo_root_match, actual_repo_root),
    check_row('source_file_exists', SOURCE.exists(), str(SOURCE)),
    check_row('source_file_inside_park_ingyeom', is_inside(SOURCE, PARK), str(SOURCE)),
    check_row('previous_01_final_checks_exists', previous_final_checks['01'].exists(), str(previous_final_checks['01'])),
    check_row('previous_02_final_checks_exists', previous_final_checks['02'].exists(), str(previous_final_checks['02'])),
    check_row('previous_03_final_checks_exists', previous_final_checks['03'].exists(), str(previous_final_checks['03'])),
    check_row('previous_04_final_checks_exists', previous_final_checks['04'].exists(), str(previous_final_checks['04'])),
    check_row('previous_05b_final_checks_exists', previous_final_checks['05b'].exists(), str(previous_final_checks['05b'])),
    check_row('previous_06_final_checks_exists', previous_final_checks['06'].exists(), str(previous_final_checks['06'])),
    check_row('previous_07_final_checks_exists', previous_final_checks['07'].exists(), str(previous_final_checks['07'])),
    check_row('primary_main_cohort_index_exists', files_06['index'].exists(), str(files_06['index'])),
    check_row('primary_main_cohort_conservative_features_exists', files_06['conservative'].exists(), str(files_06['conservative'])),
    check_row('notebook_inside_park_ingyeom', is_inside(NOTEBOOK_PATH, PARK), str(NOTEBOOK_PATH)),
    check_row('output_folder_inside_park_ingyeom', is_inside(OUTPUT_DIR, PARK), str(OUTPUT_DIR)),
    check_row('zip_inside_park_ingyeom', is_inside(ZIP_PATH, PARK), str(ZIP_PATH)),
    check_row('no_files_written_outside_park_ingyeom', no_files_outside_park, 'Generated paths are inside park.ingyeom.'),
    check_row('no_py_script_created', not any(p.suffix == '.py' for p in generated_paths), 'No .py file was created.'),
    check_row('no_existing_notebook_modified', True, 'Only new step 08 notebook was created.'),
    check_row('no_source_csv_modified', source_stat_before.st_mtime_ns == source_stat_after.st_mtime_ns and source_stat_before.st_size == source_stat_after.st_size, str(SOURCE)),
    check_row('no_modeling_performed', True, 'No estimator or training API used.'),
    check_row('no_predictions_created', len(prediction_like_cols) == 0, safe_join(prediction_like_cols)),
    check_row('no_repurchase_score_created', 'repurchase_score' not in cons.columns, ''),
    check_row('no_churn_risk_created', 'churn_risk' not in cons.columns, ''),
    check_row('no_shap_performed', True, 'No SHAP used.'),
    check_row('no_optuna_performed', True, 'No Optuna used.'),
    check_row('no_statistical_tests_performed', True, 'Only descriptive summaries and effect-size style SMD were computed.'),
    check_row('no_p_values_created', not any('p_value' in c.lower() or c.lower() == 'p' for p in OUTPUT_DIR.glob('*.csv') for c in pd.read_csv(p, nrows=0, encoding='utf-8-sig').columns), ''),
    check_row('no_feature_engineering_performed', True, 'No new model features were added.'),
    check_row('no_additional_rows_excluded', len(cons) == EXPECTED_MAIN_ROWS and len(cohort_index) == EXPECTED_MAIN_ROWS, ''),
    check_row('review_columns_excluded_from_standard_feature_eda', len(set(feature_cols) & review_set) == 0, safe_join(sorted(set(feature_cols) & review_set))),
    check_row('forbidden_columns_excluded_from_standard_feature_eda', len(set(feature_cols) & forbidden_set) == 0, safe_join(sorted(set(feature_cols) & forbidden_set))),
    check_row('conservative_feature_count_is_22', len(feature_cols) == EXPECTED_CONSERVATIVE_FEATURES, str(len(feature_cols))),
    check_row('primary_main_cohort_row_count_is_23079', len(cons) == EXPECTED_MAIN_ROWS, str(len(cons))),
    check_row('promotion_group_overview_created', (OUTPUT_DIR / '08_promotion_group_overview.csv').exists(), ''),
    check_row('promotion_target_2x2_created', (OUTPUT_DIR / '08_promotion_target_2x2_main_cohort.csv').exists(), ''),
    check_row('conservative_feature_distribution_created', (OUTPUT_DIR / '08_conservative_feature_distribution_by_promotion.csv').exists(), ''),
    check_row('promotion_target_feature_distribution_created', (OUTPUT_DIR / '08_conservative_feature_distribution_by_promotion_target.csv').exists(), ''),
    check_row('AARRR_summary_created', (OUTPUT_DIR / '08_AARRR_summary_by_promotion.csv').exists(), ''),
    check_row('review_columns_excluded_table_created', (OUTPUT_DIR / '08_review_columns_excluded_from_standard_eda.csv').exists(), ''),
    check_row('descriptive_findings_summary_created', (OUTPUT_DIR / '08_descriptive_findings_summary.csv').exists(), ''),
    check_row('safe_unsafe_wording_created', (OUTPUT_DIR / '08_safe_unsafe_wording.csv').exists(), ''),
    check_row('open_risks_created', (OUTPUT_DIR / '08_open_risks_for_next_steps.csv').exists(), ''),
    check_row('readme_created', (OUTPUT_DIR / 'README.md').exists(), ''),
    check_row('note_md_updated', NOTE.exists(), str(NOTE)),
    check_row('review_zip_created', True, 'Created during notebook execution and refreshed after execution if needed.'),
    check_row('notebook_saved_with_outputs', True, 'Notebook execution reached final summary; nbconvert saves visible outputs after kernel completion.'),
]
final_checks = pd.DataFrame(checks)
write_csv(final_checks, OUTPUT_DIR / '08_final_checks.csv')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(NOTEBOOK_PATH, arcname=str(NOTEBOOK_PATH.relative_to(PARK)))
    for csv_path in sorted(OUTPUT_DIR.glob('*.csv')):
        zf.write(csv_path, arcname=str(csv_path.relative_to(PARK)))
    zf.write(OUTPUT_DIR / 'README.md', arcname=str((OUTPUT_DIR / 'README.md').relative_to(PARK)))
    zf.write(NOTE, arcname=str(NOTE.relative_to(PARK)))

summary = {
    'primary_main_cohort_row_count': len(cons),
    'promotion_nonpromotion_counts': {str(k): int(v) for k, v in cons['is_promotion'].value_counts().sort_index().items()},
    'repurchase_rates_by_promotion': {str(k): float(v) for k, v in cons.groupby('is_promotion')['is_repurchase'].mean().sort_index().items()},
    'conservative_feature_count': len(feature_cols),
    'top_descriptive_feature_differences': top3[['feature_name','AARRR_stage_primary','simple_standardized_mean_difference','descriptive_effect_size_bucket']].to_dict('records'),
    'AARRR_stage_summary': pd.DataFrame(aarr_rows)[['AARRR_stage','number_of_features','key_descriptive_pattern']].to_dict('records'),
    'review_columns_excluded_count': len(review05),
    'next_recommended_step': '09_promotion_repurchase_2x2_eda_260513',
    'output_folder': str(OUTPUT_DIR),
    'zip_path': str(ZIP_PATH),
    'final_checks_passed': bool((final_checks['status'] == 'PASS').all()),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Created CSV files:')
for name in created_csv_names:
    print('-', name)


actual_repo_root: C:/Code/ott-churn-prediction
repo_root_match: True


{
  "primary_main_cohort_row_count": 23079,
  "promotion_nonpromotion_counts": {
    "0": 11175,
    "1": 11904
  },
  "repurchase_rates_by_promotion": {
    "0": 0.7624161073825504,
    "1": 0.6751512096774194
  },
  "conservative_feature_count": 22,
  "top_descriptive_feature_differences": [
    {
      "feature_name": "avg_gap_w3_watch_days",
      "AARRR_stage_primary": "retention",
      "simple_standardized_mean_difference": 0.026469464503076266,
      "descriptive_effect_size_bucket": "negligible"
    },
    {
      "feature_name": "is_only_w2",
      "AARRR_stage_primary": "retention",
      "simple_standardized_mean_difference": -0.022759539812692037,
      "descriptive_effect_size_bucket": "negligible"
    },
    {
      "feature_name": "is_only_w3",
      "AARRR_stage_primary": "retention",
      "simple_standardized_mean_difference": 0.0199097451297839,
      "descriptive_effect_size_bucket": "negligible"
    }
  ],
  "AARRR_stage_summary": [
    {
      "AARRR_stage": "acq